In [1]:
!pip install pyspark

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder.getOrCreate()

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Utiliza-se a função "sep" para o spark conseguir ler o arquivo csv como "," e não ";" que foi a forma que o arquivo foi salvo
pessoas = spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true").csv("drive/MyDrive/Colab Notebooks/spark/data/pessoas_consolidado.csv", header=True, inferSchema=True)
sinistros = spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true").csv("drive/MyDrive/Colab Notebooks/spark/data/sinistros_consolidado.csv", header=True, inferSchema=True)
veiculos = spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true").csv("drive/MyDrive/Colab Notebooks/spark/data/veiculos_consolidado.csv", header=True, inferSchema=True)

In [6]:
pessoas.show(5)
pessoas.printSchema()
sinistros.show(5)
sinistros.printSchema()
veiculos.show(5)
veiculos.printSchema()

+-----------+---------+----------+--------+-----------+---------------------+-------------------+-------------------+--------------+----------------------+---------+-----+---------------+------------------------+------------------+--------------+-----------------+--------------------+-------------+------------+------------+------------+----------------+----------+---------+---------+---------+-------------+-----------+---------+--------------------+
|id_sinistro|id_pessoa|id_veiculo|cod_ibge|  municipio|regiao_administrativa|           tipo_via|tipo_veiculo_vitima|tipo_de_vitima|modo_transporte_vitima|     sexo|idade|gravidade_lesao|faixa_etaria_demografica|faixa_etaria_legal|     profissao|grau_de_instrucao|       nacionalidade|data_sinistro|ano_sinistro|mes_sinistro|dia_sinistro|ano_mes_sinistro|data_obito|ano_obito|mes_obito|dia_obito|ano_mes_obito|local_obito|local_via|tempo_sinistro_obito|
+-----------+---------+----------+--------+-----------+---------------------+---------------

In [7]:
print('Quantidade de dados pessoas: ', (pessoas.count(), len(pessoas.columns)))
print('Quantidade de dados sinistros: ', (sinistros.count(), len(sinistros.columns)))
print('Quantidade de dados veiculos: ', (veiculos.count(), len(veiculos.columns)))

Quantidade de dados pessoas:  (59171, 31)
Quantidade de dados sinistros:  (44846, 50)
Quantidade de dados veiculos:  (50610, 12)


In [8]:
print('Total de pessoas: ', pessoas.count())

pessoas_unicas = pessoas.dropDuplicates()
print('Total de valores unicos: ', pessoas_unicas.count())

pessoas_remocao_nulos = pessoas_unicas.na.drop()
print('Total de valores nulos: ', pessoas_remocao_nulos.count())

pessoas_remocao_nulos_id = pessoas_unicas.na.drop(subset=['id_sinistro', 'id_pessoa'])
print('Total de valores após a limpeza de id nulos: ', pessoas_remocao_nulos_id.count())

Total de pessoas:  59171
Total de valores unicos:  59171
Total de valores nulos:  0
Total de valores após a limpeza de id nulos:  59171


In [9]:
print('Total de veiculos: ', veiculos.count())

veiculos_unicos = veiculos.dropDuplicates()
print('Total de valores unicos: ', veiculos_unicos.count())

veiculos_remocao_nulos = veiculos_unicos.na.drop()
print('Total de dados após a limpeza de valores nulos: ', veiculos_remocao_nulos.count())

veiculos_remocao_nulos_id = veiculos_unicos.na.drop(subset=['id_sinistro', 'id_veiculo'])
print('Total de valores nulos após a limpeza de id nulos: ', veiculos_remocao_nulos_id.count())

Total de veiculos:  50610
Total de valores unicos:  50610
Total de dados após a limpeza de valores nulos:  42750
Total de valores nulos após a limpeza de id nulos:  50610


In [10]:
print('Total de sinistros: ', sinistros.count())

sinistros_unicos = sinistros.dropDuplicates()
print('Total de valores unicos: ', sinistros_unicos.count())

sinistros_remocao_nulos = sinistros_unicos.na.drop()
print('Total de valores nulos: ', sinistros_remocao_nulos.count())

sinistros_remocao_nulos_id = sinistros_unicos.na.drop(subset=['id_sinistro'])
print('Total de valores nulos após a limpeza de id nulos: ', sinistros_remocao_nulos_id.count())

Total de sinistros:  44846
Total de valores unicos:  44846
Total de valores nulos:  0
Total de valores nulos após a limpeza de id nulos:  44846


In [11]:
# Cast para conversão de dados das colunas
pessoas_tratados_df = pessoas. \
  withColumn('id_sinistro', col('id_sinistro').cast('string')). \
  withColumn('id_pessoa', col('id_pessoa').cast('string')). \
  withColumn('id_veiculo', col('id_veiculo').cast('string')). \
  withColumn('idade', col('idade').cast('int'))
pessoas_tratados_df.printSchema()
pessoas_tratados_df.show(5)

root
 |-- id_sinistro: string (nullable = true)
 |-- id_pessoa: string (nullable = true)
 |-- id_veiculo: string (nullable = true)
 |-- cod_ibge: integer (nullable = true)
 |-- municipio: string (nullable = true)
 |-- regiao_administrativa: string (nullable = true)
 |-- tipo_via: string (nullable = true)
 |-- tipo_veiculo_vitima: string (nullable = true)
 |-- tipo_de_vitima: string (nullable = true)
 |-- modo_transporte_vitima: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- gravidade_lesao: string (nullable = true)
 |-- faixa_etaria_demografica: string (nullable = true)
 |-- faixa_etaria_legal: string (nullable = true)
 |-- profissao: string (nullable = true)
 |-- grau_de_instrucao: string (nullable = true)
 |-- nacionalidade: string (nullable = true)
 |-- data_sinistro: string (nullable = true)
 |-- ano_sinistro: integer (nullable = true)
 |-- mes_sinistro: integer (nullable = true)
 |-- dia_sinistro: integer (nullable = true)
 

In [12]:
sinistros_tratados_df = sinistros. \
  withColumn('id_sinistro', col('id_sinistro').cast('string')). \
  withColumn('numero_logradouro', col('numero_logradouro').cast('int')). \
  withColumn('qtd_pedestre', col('qtd_pedestre').cast('int')). \
  withColumn('qtd_bicicleta', col('qtd_bicicleta').cast('int')). \
  withColumn('qtd_motocicleta', col('qtd_motocicleta').cast('int')). \
  withColumn('qtd_automovel', col('qtd_automovel').cast('int')). \
  withColumn('qtd_onibus', col('qtd_onibus').cast('int')). \
  withColumn('qtd_caminhao', col('qtd_caminhao').cast('int')). \
  withColumn('qtd_veic_outros', col('qtd_veic_outros').cast('int')). \
  withColumn('qtd_veic_nao_disponivel', col('qtd_veic_nao_disponivel').cast('int')). \
  withColumn('qtd_gravidade_fatal', col('qtd_gravidade_fatal').cast('int')). \
  withColumn('qtd_gravidade_grave', col('qtd_gravidade_grave').cast('int')). \
  withColumn('qtd_gravidade_leve', col('qtd_gravidade_leve').cast('int')). \
  withColumn('qtd_gravidade_ileso', col('qtd_gravidade_ileso').cast('int')). \
  withColumn('qtd_gravidade_nao_disponivel', col('qtd_gravidade_nao_disponivel').cast('int'))
sinistros_tratados_df.printSchema()
sinistros_tratados_df.show(5)

root
 |-- id_sinistro: string (nullable = true)
 |-- tipo_registro: string (nullable = true)
 |-- data_sinistro: string (nullable = true)
 |-- ano_sinistro: integer (nullable = true)
 |-- mes_sinistro: integer (nullable = true)
 |-- dia_sinistro: integer (nullable = true)
 |-- hora_sinistro: timestamp (nullable = true)
 |-- ano_mes_sinistro: string (nullable = true)
 |-- dia_da_semana: string (nullable = true)
 |-- turno: string (nullable = true)
 |-- logradouro: string (nullable = true)
 |-- numero_logradouro: integer (nullable = true)
 |-- tipo_via: string (nullable = true)
 |-- tipo_local: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- cod_ibge: integer (nullable = true)
 |-- municipio: string (nullable = true)
 |-- regiao_administrativa: string (nullable = true)
 |-- administracao: string (nullable = true)
 |-- conservacao: string (nullable = true)
 |-- circunscricao: string (nullable = true)
 |-- tp_sinistro_primario:

In [13]:
veiculos_tratados_df = veiculos. \
  withColumn('id_sinistro', col('id_sinistro').cast('string')). \
  withColumn('id_veiculo', col('id_veiculo').cast('string')). \
  withColumn('ano_fab', col('ano_fab').cast('int')). \
  withColumn('ano_modelo', col('ano_modelo').cast('int'))
veiculos_tratados_df.printSchema()
veiculos_tratados_df.show(5)

root
 |-- id_sinistro: string (nullable = true)
 |-- id_veiculo: string (nullable = true)
 |-- marca_modelo: string (nullable = true)
 |-- ano_fab: integer (nullable = true)
 |-- ano_modelo: integer (nullable = true)
 |-- cor_veiculo: string (nullable = true)
 |-- tipo_veiculo: string (nullable = true)
 |-- data_sinistro: string (nullable = true)
 |-- ano_sinistro: integer (nullable = true)
 |-- mes_sinistro: integer (nullable = true)
 |-- dia_sinistro: integer (nullable = true)
 |-- ano_mes_sinistro: string (nullable = true)

+-----------+----------+------------+-------+----------+-----------+--------------+-------------+------------+------------+------------+----------------+
|id_sinistro|id_veiculo|marca_modelo|ano_fab|ano_modelo|cor_veiculo|  tipo_veiculo|data_sinistro|ano_sinistro|mes_sinistro|dia_sinistro|ano_mes_sinistro|
+-----------+----------+------------+-------+----------+-----------+--------------+-------------+------------+------------+------------+----------------+
|    

In [14]:
pessoas_tratados_df.describe().show()
sinistros_tratados_df.describe().show()
veiculos_tratados_df.describe().show()

+-------+------------------+------------------+------------------+------------------+----------+---------------------+-------------------+-------------------+--------------+----------------------+--------------+------------------+---------------+------------------------+------------------+-------------+-----------------+--------------------+-------------+------------+------------------+------------------+----------------+----------+---------+------------------+------------------+-------------+--------------------+--------------+--------------------+
|summary|       id_sinistro|         id_pessoa|        id_veiculo|          cod_ibge| municipio|regiao_administrativa|           tipo_via|tipo_veiculo_vitima|tipo_de_vitima|modo_transporte_vitima|          sexo|             idade|gravidade_lesao|faixa_etaria_demografica|faixa_etaria_legal|    profissao|grau_de_instrucao|       nacionalidade|data_sinistro|ano_sinistro|      mes_sinistro|      dia_sinistro|ano_mes_sinistro|data_obito|ano_obit

In [15]:
sinistros.groupBy("turno") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
sinistros.groupBy("dia_da_semana") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
sinistros.groupBy("tipo_via") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
sinistros.groupBy("municipio") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
sinistros.groupBy("ano_mes_sinistro") \
    .count() \
    .orderBy("ano_mes_sinistro") \
    .show()
sinistros.groupBy("tp_sinistro_primario") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+--------------+-----+
|         turno|count|
+--------------+-----+
|         TARDE|15097|
|         NOITE|13688|
|         MANHA|11952|
|     MADRUGADA| 4069|
|NAO DISPONIVEL|   40|
+--------------+-----+

+-------------+-----+
|dia_da_semana|count|
+-------------+-----+
|  Sexta-feira| 7662|
|       Sábado| 6670|
| Quarta-feira| 6261|
|Segunda-feira| 6237|
| Quinta-feira| 6142|
|  Terça-feira| 5951|
|      Domingo| 5923|
+-------------+-----+

+-------------------+-----+
|           tipo_via|count|
+-------------------+-----+
|       VIAS URBANAS|37120|
|ESTRADAS E RODOVIAS| 7651|
|     NAO DISPONIVEL|   75|
+-------------------+-----+

+--------------------+-----+
|           municipio|count|
+--------------------+-----+
|           SAO PAULO|12027|
|           GUARULHOS| 1352|
|      RIBEIRAO PRETO| 1115|
|            CAMPINAS|  973|
|SAO JOSE DO RIO P...|  822|
|SAO BERNARDO DO C...|  792|
| SAO JOSE DOS CAMPOS|  727|
|         SANTO ANDRE|  658|
|            SOROCABA|  636|
|   

In [16]:
pessoas.groupBy("sexo") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
pessoas.groupBy("tipo_de_vitima") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
pessoas.groupBy("modo_transporte_vitima") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
pessoas.groupBy("gravidade_lesao") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()
pessoas.groupBy("faixa_etaria_demografica") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+--------------+-----+
|          sexo|count|
+--------------+-----+
|     MASCULINO|43177|
|      FEMININO|15959|
|NAO DISPONIVEL|   35|
+--------------+-----+

+--------------+-----+
|tipo_de_vitima|count|
+--------------+-----+
|      CONDUTOR|29376|
|    PASSAGEIRO|22552|
|NAO DISPONIVEL| 4219|
|      PEDESTRE| 3024|
+--------------+-----+

+----------------------+-----+
|modo_transporte_vitima|count|
+----------------------+-----+
|             AUTOMOVEL|24097|
|           MOTOCICLETA|23150|
|                  NULL| 4959|
|              PEDESTRE| 3024|
|              CAMINHAO| 1659|
|                ONIBUS| 1312|
|                OUTROS|  855|
|             BICICLETA|   96|
|        NAO DISPONIVEL|   19|
+----------------------+-----+

+---------------+-----+
|gravidade_lesao|count|
+---------------+-----+
|           LEVE|28396|
|          ILESO|24984|
|          GRAVE| 3228|
|          FATAL| 1486|
| NAO DISPONIVEL| 1077|
+---------------+-----+

+------------------------+-----+

In [17]:
veiculos.groupBy("tipo_veiculo") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+--------------+-----+
|  tipo_veiculo|count|
+--------------+-----+
|     AUTOMOVEL|22625|
|   MOTOCICLETA|20786|
|      CAMINHAO| 1962|
|        OUTROS| 1634|
|NAO DISPONIVEL| 1388|
|        ONIBUS| 1179|
|     BICICLETA| 1036|
+--------------+-----+



In [18]:
sinistros_coalesce_df = sinistros_tratados_df.coalesce(1)
pessoas_coalesce_df = pessoas_tratados_df.coalesce(1)
veiculos_coalesce_df = veiculos_tratados_df.coalesce(1)
print("Número de partições após coalesce (sinistros): ", sinistros_coalesce_df.rdd.getNumPartitions())
print("Número de partições após coalesce (pessoas): ", pessoas_coalesce_df.rdd.getNumPartitions())
print("Número de partições após coalesce (veículos): ", veiculos_coalesce_df.rdd.getNumPartitions())

Número de partições após coalesce (sinistros):  1
Número de partições após coalesce (pessoas):  1
Número de partições após coalesce (veículos):  1


In [19]:
sinistros_tratados_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv('output/sinistros_consolidado_csv')
pessoas_tratados_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv('output/pessoas_consolidado_csv')
veiculos_tratados_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv('output/veiculos_consolidado_csv')

In [20]:
spark.read.option("header", "true").csv("output/sinistros_consolidado_csv").show(5)
spark.read.option("header", "true").csv("output/pessoas_consolidado_csv").show(5)
spark.read.option("header", "true").csv("output/veiculos_consolidado_csv").show(5)

+-----------+------------------+-------------+------------+------------+------------+--------------------+----------------+-------------+-----+--------------------+-----------------+-------------------+--------------+--------------+--------------+--------+--------------------+---------------------+--------------+-----------+-------------+--------------------+------------+-------------+---------------+-------------+----------+------------+---------------+-----------------------+-------------------+-------------------+------------------+-------------------+----------------------------+--------------------------+----------------------------------+---------------------------+----------------------------+---------------------------+-------------------------------+--------------------------+------------------+------------------------+-----------------------+-------------------------+----------------------+------------------+--------------------------+
|id_sinistro|     tipo_registro|data_sin

In [21]:
spark.stop()